# Stiffness 4 — the particle

A general rate model on the same lab column, with a **linear isotherm at rapid equilibrium**. The isotherm is linear so that no curvature can sharpen a front, and algebraic so that no binding rate adds a time scale — both were shown in earlier notebooks to drown out whatever else is going on.

What is left is a particle: a film resistance, a pore diffusion time scale, and a **radial grid of its own**.

| | |
|---|---|
| column | as before: 0.1 m, 150 cm/h, Péclet ≈ 1100, 100 axial cells |
| particle radius | 45 µm, porosity 0.75 |
| film diffusion | 10⁻⁵ m/s → 1.14 s⁻¹ |
| pore diffusion | 10⁻¹¹ m²/s — a protein in a resin pore |
| isotherm | linear, $q = 10\,c$, rapid equilibrium |

While the bulk concentration changes over a time $t$, diffusion reaches about $\delta \approx \sqrt{D_p t}$ into the particle. Compared with the particle radius that says whether the particle is uniform or carries a thin shell over an untouched core; compared with the **radial cell size** it says whether the grid can represent it. Those are the two knobs.

In [ ]:
# This notebook lives one folder below the workshop root, while `utility`
# and everything under `data/` are addressed relative to that root.
# Step up into it before importing anything.
import os
import sys

root = os.path.abspath(os.curdir)
while not os.path.isdir(os.path.join(root, 'utility')):
    root, previous = os.path.dirname(root), root
    if root == previous:
        raise RuntimeError('no folder containing utility/ above this notebook')

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

In [ ]:
import utility.particle_explorer as explorer

explorer.show_setting()

## What to watch

Note the rate table: the particle grid contributes a row of its own, `pore diffusion, one particle cell`, which grows as $D_p/\Delta r^2$. Refining the particle manufactures a fast process exactly as refining the axial grid does.

So this study sweeps the **pore diffusivity** and the **number of radial cells**, and reports three things that usually agree and here do not:

- the **stiffness indicator**, that fast radial rate over the slowest process in the model;
- the **step count**;
- the **error** against a converged particle grid, because this is a case where resolution changes the answer as well as the cost.

Sweep both and watch whether the indicator predicts anything at all.

In [ ]:
from ipywidgets import interact

show_particle = explorer.make_explorer()

interact(show_particle, **show_particle.sliders);

## What happened

**The stiffness indicator points the wrong way.** Refining the particle from 1 to 32 radial cells multiplies it by a thousand — and the step count *falls* by half while the answer improves five hundredfold:

| particle cells | stiffness indicator | BDF steps | error | seconds |
|---|---|---|---|---|
| 1 | 4.9·10² | **2920** | 2.89 % | 0.33 |
| 8 | 3.2·10⁴ | 1640 | 0.10 % | 0.79 |
| 32 | 5.1·10⁵ | **1379** | 0.006 % | 2.60 |

The pore diffusivity knob says the same thing backwards. At 8 radial cells, going from $D_p = 10^{-9}$ to $10^{-14}$ drops the indicator through five decades, from 3.2·10⁶ to 64 — and the step count *rises*, 1458 → 2263.

**What does predict the cost is the error.** Across the whole table the step count tracks how badly the radial profile is resolved, not how stiff the model is. An under-resolved particle is both wrong and slow; a well-resolved one is right and cheap, and costs only in degrees of freedom.

**And the particle is cheap in any case.** The widest spread in the whole sweep is about a factor of two, against the factor of **14** that a steep axial front cost in the previous notebook.

<details>
<summary><b>Click to learn more: why the particle cannot be expensive</b></summary>

The radial direction contains only diffusion. The axial direction contains convection as well, and that is the whole difference.

Diffusion smooths; it can never steepen a profile. Whatever shape the particle starts with, $\delta \approx \sqrt{D_p t}$ grows, and the gradient falls. Convection does the opposite: it carries a front along and, with a nonlinear isotherm, actively sharpens it. The steep front study cost a factor of 14 because an advective front stays sharp while the grid tries to follow it.

So the worst a particle can do is present a thin shell — a feature that is sharp in space but is smoothing with time and carries little material once $D_p$ is small enough to make it thin. That is worth a factor of two, and the factor of two is under-resolution rather than stiffness.

</details>

> **Take-away.** Do not economise on radial cells. A coarse particle grid is the one choice in this chapter that is wrong *and* slow at the same time — it costs accuracy and step count together, and saves you only degrees of freedom. And if you are reaching for a stiffness ratio to decide whether a model will be expensive: here it would have told you the exact opposite of the truth.